In [13]:
import os
import re
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

bot_token = (os.getenv("TELEGRAM_BOT_TOKEN") or "").strip()
chat_id = (os.getenv("TELEGRAM_CHAT_ID") or "").strip()

if not bot_token or not chat_id:
    print("❌ Mangler TELEGRAM_BOT_TOKEN eller TELEGRAM_CHAT_ID i .env")
elif re.fullmatch(r"[0-9a-f]{32}", bot_token):
    print("⚠️ Detektert Telegram 'api_hash' (32-tegns hex fra my.telegram.org) i stedet for Bot Token!")
    print(f"   • TELEGRAM_BOT_TOKEN={bot_token} er en MTProto api_hash ( og {chat_id} er api_id).")
    print("   • REST-endepunktet https://api.telegram.org/bot<TOKEN>/sendMessage krever en Bot-token fra @BotFather (med kolon, f.eks. 7123456789:AAH...).")
    print("\nSlik fikser du det på 30 sekunder i Telegram-appen:")
    print("   1. Søk opp @BotFather i Telegram -> send /newbot (eller /token hvis du allerede har en bot).")
    print("   2. Kopier tokenet (format: 1234567890:ABCdef...) inn i TELEGRAM_BOT_TOKEN i .env.")
    print("   3. Send en kort melding (f.eks. 'Hei') til botten din i Telegram, og kjør denne cellen på nytt!")
else:
    # Hvis brukeren har satt inn gyldig @BotFather-token, sjekk også om vi kan hente riktig chat_id automatisk via getUpdates
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {
        "chat_id": chat_id,
        "text": "🟢 *AI Trading Agent Tilkoblet!*\nTelegram HITL-varsling er operativ.",
        "parse_mode": "Markdown"
    }
    resp = requests.post(url, json=payload, timeout=10)
    if resp.status_code == 200:
        print("✅ Telegram-melding levert til mobilen din!")
    else:
        print(f"🔴 Feil ({resp.status_code}):", resp.text)
        # Hjelp brukeren med å finne riktig TELEGRAM_CHAT_ID automatisk fra siste melding til botten
        upd = requests.get(f"https://api.telegram.org/bot{bot_token}/getUpdates", timeout=10).json()
        results = upd.get("result", [])
        if results:
            detected_chat_id = results[-1].get("message", {}).get("chat", {}).get("id")
            print(f"💡 Tips: Fant din faktiske TELEGRAM_CHAT_ID fra siste melding til botten: {detected_chat_id}")
            print(f"   Oppdater TELEGRAM_CHAT_ID={detected_chat_id} i .env og kjør på nytt.")

✅ Telegram-melding levert til mobilen din!
